# Customer Segmentation using K-Means Clustering
## Behavior-Based Customer Profiling

**Objective:** Identify groups of customers with similar income and spending behavior using K-Means, validate the clustering structure, profile the segments, and translate them into business strategies.

**Core story:** Customer Data → EDA → Feature Selection → Elbow Method → Silhouette Validation → K-Means → Cluster Profiling → Behavioral Feature → Customer Personas → Business Recommendations → New Customer Prediction

**Important:** K-Means uses **Annual Income (k$)** and **Spending Score (1-100)**. The Spending-to-Income Ratio is used only for post-clustering profiling.

## 1. Import Libraries
We use Pandas/NumPy for data handling, Matplotlib/Seaborn for visualization, and Scikit-learn for K-Means, scaling, and Silhouette Score.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

## 2. Load the Dataset
The project uses the Mall Customers dataset. The code first tries the project GitHub copy; if unavailable, it allows a CSV upload in Colab.

In [ ]:
url = "https://raw.githubusercontent.com/luvmanchanda/Customer-Segmentation-Using-K-Means-Clustering/main/Mall_Customers.csv"

try:
    customer_data = pd.read_csv(url)
except Exception:
    from google.colab import files
    uploaded = files.upload()
    customer_data = pd.read_csv(next(iter(uploaded)))

print("Dataset Shape:", customer_data.shape)
customer_data.head()

## 3. Basic Data Understanding
Check shape, data types, missing values, duplicates, and summary statistics before modeling.

In [ ]:
print("Shape:", customer_data.shape)
print("\nData Types:")
print(customer_data.dtypes)
print("\nMissing Values:")
print(customer_data.isnull().sum())
print("\nDuplicate Rows:", customer_data.duplicated().sum())
print("\nSummary Statistics:")
display(customer_data.describe())

## 4. Exploratory Data Analysis (EDA)
**EDA (Exploratory Data Analysis)** means statistically and visually exploring data to understand distributions, relationships, patterns, and possible issues.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(customer_data["Age"], kde=True, ax=axes[0])
axes[0].set_title("Age Distribution")

sns.histplot(customer_data["Annual Income (k$)"], kde=True, ax=axes[1])
axes[1].set_title("Annual Income Distribution")

sns.histplot(customer_data["Spending Score (1-100)"], kde=True, ax=axes[2])
axes[2].set_title("Spending Score Distribution")

plt.tight_layout()
plt.show()

## 5. Income vs Spending Behavior
This is the key relationship for our segmentation objective.

In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=customer_data,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    s=90
)
plt.title("Annual Income vs Spending Score")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.show()

## 6. Feature Selection
We select **Annual Income** and **Spending Score** because they represent purchasing capacity and spending behavior.

`CustomerID` is only an identifier. `Gender` and `Age` are retained for profiling but are not used in the core clustering model.

In [ ]:
features = ["Annual Income (k$)", "Spending Score (1-100)"]
X = customer_data[features].copy()
X.head()

## 7. Feature Scaling Check
K-Means is distance-based, so feature scale matters. These two variables are already on reasonably comparable numeric scales, so the main model remains on the original interpretable units. We also demonstrate standardization as a robustness consideration.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Original feature ranges:")
display(X.describe().loc[["min", "max"]])
print("\nScaled feature means:", X_scaled.mean(axis=0))
print("Scaled feature standard deviations:", X_scaled.std(axis=0))

## 8. Elbow Method

**WCSS (Within-Cluster Sum of Squares)** measures the total squared distance of observations from the centroid of their assigned cluster.

$$WCSS = \sum_{k=1}^{K} \sum_{x_i \in C_k} \|x_i-\mu_k\|^2$$

Where `K` is the number of clusters, `C_k` is the kth cluster, `x_i` is an observation, and `μ_k` is the kth centroid.

WCSS decreases as K increases, so we look for an **elbow**, where additional clusters provide diminishing improvement.

In [ ]:
wcss = []

for k in range(1, 11):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X)
    wcss.append(model.inertia_)

plt.figure(figsize=(9, 6))
plt.plot(range(1, 11), wcss, marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("WCSS")
plt.title("Elbow Method")
plt.xticks(range(1, 11))
plt.show()

## 9. Silhouette Validation

**Silhouette Score** evaluates both:
- **Cohesion:** how close an observation is to its own cluster.
- **Separation:** how far it is from the nearest other cluster.

The score ranges approximately from **-1 to +1**:
- near +1 → well-separated/cohesive
- around 0 → near a cluster boundary
- negative → possible poor assignment

We calculate it for several K values. This is a validation step, not something we calculate and then ignore.

In [ ]:
silhouette_scores = []
k_values = list(range(2, 11))

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X)
    score = silhouette_score(X, labels)
    silhouette_scores.append(score)
    print(f"K = {k}, Silhouette Score = {score:.4f}")

plt.figure(figsize=(9, 6))
plt.plot(k_values, silhouette_scores, marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for Different K Values")
plt.xticks(k_values)
plt.show()

best_k_by_silhouette = k_values[int(np.argmax(silhouette_scores))]
print(f"Best K according to Silhouette Score: {best_k_by_silhouette}")
print(f"Best Silhouette Score: {max(silhouette_scores):.4f}")

## 10. Selecting K
The Elbow Method evaluates **compactness**, while Silhouette Score evaluates **cohesion and separation**.

For this project we retain **K = 5** as the selected business segmentation structure, supported by the diagnostics and interpretability. We do not claim that K=5 is chosen from silhouette alone.

In [ ]:
selected_k = 5

k5_index = k_values.index(selected_k)
k5_silhouette = silhouette_scores[k5_index]

print(f"Selected K: {selected_k}")
print(f"Silhouette Score for K={selected_k}: {k5_silhouette:.4f}")

## 11. Train the Final K-Means Model
K-Means repeatedly assigns observations to the nearest centroid, recalculates centroids, and repeats until convergence.

In [ ]:
kmeans = KMeans(n_clusters=selected_k, random_state=42, n_init=10)
customer_data["Cluster"] = kmeans.fit_predict(X)

print("Cluster Counts:")
print(customer_data["Cluster"].value_counts().sort_index())

## 12. Visualize Final Clusters
Different colors represent clusters and X markers represent centroids.

In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=customer_data,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    hue="Cluster",
    palette="tab10",
    s=90
)

centers = kmeans.cluster_centers_
plt.scatter(
    centers[:, 0], centers[:, 1],
    marker="X", s=250, color="black",
    label="Centroids"
)

plt.title("Customer Segmentation using K-Means")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.legend()
plt.show()

## 13. Customer Profiling
**Customer Profiling** means summarizing the characteristics and behavior of each customer segment.

We calculate customer count, average age, income, and spending score.

In [ ]:
cluster_profile = (
    customer_data.groupby("Cluster")
    .agg(
        Customers=("CustomerID", "count"),
        Avg_Age=("Age", "mean"),
        Avg_Income=("Annual Income (k$)", "mean"),
        Avg_Spending=("Spending Score (1-100)", "mean")
    )
    .round(2)
)

cluster_profile

## 14. Behavioral Feature Engineering
We create a post-clustering behavioral indicator:

$$Spending\text{-}to\text{-}Income\ Ratio =
\frac{Spending\ Score}{Annual\ Income}$$

This gives a relative measure of spending compared with income.

**Important:** it is used for profiling/interpretation, not as a K-Means input feature.

In [ ]:
customer_data["Spending_to_Income_Ratio"] = (
    customer_data["Spending Score (1-100)"]
    / customer_data["Annual Income (k$)"]
)

ratio_profile = (
    customer_data.groupby("Cluster")["Spending_to_Income_Ratio"]
    .mean()
    .round(3)
    .rename("Avg_Spending_to_Income_Ratio")
)

cluster_profile = cluster_profile.join(ratio_profile)
cluster_profile

## 15. Create Customer Personas
Cluster numbers have no inherent business meaning. We interpret clusters using their average income and spending characteristics and assign descriptive personas.

In [ ]:
income_median = cluster_profile["Avg_Income"].median()
spending_median = cluster_profile["Avg_Spending"].median()

def assign_segment(row):
    high_income = row["Avg_Income"] >= income_median
    high_spending = row["Avg_Spending"] >= spending_median

    if high_income and high_spending:
        return "Premium / High-Value Customers"
    elif high_income and not high_spending:
        return "High-Income Low-Spending Customers"
    elif not high_income and high_spending:
        return "Budget-Conscious Active Customers"
    else:
        return "Low-Engagement Customers"

cluster_profile["Customer_Segment"] = cluster_profile.apply(assign_segment, axis=1)
cluster_profile

## 16. Map Segment Names to Customers
We add the business-oriented segment name to every individual customer.

In [ ]:
customer_data["Customer_Segment"] = customer_data["Cluster"].map(
    cluster_profile["Customer_Segment"]
)

customer_data.head()

## 17. Behavior-Based Customer Segmentation Visualization

In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=customer_data,
    x="Annual Income (k$)",
    y="Spending Score (1-100)",
    hue="Customer_Segment",
    s=90
)
plt.title("Behavior-Based Customer Segmentation")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.show()

## 18. Business Interpretation

- **Premium / High-Value Customers:** high income + high spending → VIP offers, loyalty programs, premium products.
- **High-Income Low-Spending Customers:** high purchasing capacity but lower spending → personalized promotions and engagement campaigns.
- **Budget-Conscious Active Customers:** lower income but relatively strong spending → discounts, rewards, value-based offers.
- **Low-Engagement Customers:** lower income + lower spending → re-engagement or low-cost targeted campaigns.

The exact assignment is read from the generated `cluster_profile` table.

## 19. New Customer Prediction
A new customer can be assigned to an existing cluster using the trained K-Means model, after which the cluster is mapped to its customer persona.

In [ ]:
new_customer = pd.DataFrame({
    "Annual Income (k$)": [120],
    "Spending Score (1-100)": [95]
})

predicted_cluster = kmeans.predict(new_customer)[0]
predicted_segment = cluster_profile.loc[predicted_cluster, "Customer_Segment"]

print("Predicted Cluster:", predicted_cluster)
print("Predicted Customer Segment:", predicted_segment)

## 20. Final Validation Summary
This gives the final WCSS and Silhouette Score for the selected K=5 model.

In [ ]:
final_wcss = kmeans.inertia_
final_silhouette = silhouette_score(X, customer_data["Cluster"])

print("===== FINAL MODEL SUMMARY =====")
print(f"Selected K: {selected_k}")
print(f"Final WCSS: {final_wcss:.4f}")
print(f"Final Silhouette Score: {final_silhouette:.4f}")

# 21. Complete End-to-End Workflow

```text
Customer Dataset
        ↓
Data Cleaning + Basic Checks
        ↓
Exploratory Data Analysis (EDA)
        ↓
Understand Income & Spending Behavior
        ↓
Select Annual Income + Spending Score
        ↓
Elbow Method using WCSS
        ↓
Evaluate Multiple K Values
        ↓
Silhouette Score Validation
        ↓
Select K = 5
        ↓
Train K-Means
        ↓
Generate Customer Clusters
        ↓
Cluster Visualization
        ↓
Add Cluster Labels
        ↓
Behavioral Feature Engineering
(Spending-to-Income Ratio)
        ↓
Cluster Profiling
        ↓
Customer Personas
        ↓
Business Recommendations
        ↓
New Customer Prediction
        ↓
Predicted Segment + Strategy
```

### Final Project Story

> **I used K-Means clustering to identify customer groups based on annual income and spending behavior. I first performed EDA and selected the two main clustering features. I used the Elbow Method to examine cluster compactness and Silhouette Score to validate cluster cohesion and separation, then retained five clusters. Instead of stopping at clustering, I added a behavior-based customer profiling layer, including a spending-to-income indicator, to interpret the clusters as customer personas. I then translated these segments into targeted business strategies and demonstrated how a new customer can be assigned to an existing segment.**